# Check if the coding sheet is the most updated one

Before doing this part, please create a blank cohort_coding sheet. Copy and paste the sheet ID to the 4th cell (cohort coding)

In [ ]:
# Install pygsheets
!pip install --upgrade -q pygsheets

In [ ]:
# --- Colab setup: clone the repo so code/ is available locally ---
# Skip/edit this cell if you're already running inside a cloned copy of the
# repo (e.g. running locally, or you mounted/cloned it another way).
import os

REPO_URL = 'https://github.com/<your-org>/<your-repo>.git'  # <- set this
REPO_DIR = '<your-repo>'  # <- local folder name git clone will create

if not os.path.isdir(REPO_DIR):
    !git clone {REPO_URL}

os.chdir(REPO_DIR)
print('cwd:', os.getcwd())


In [ ]:
# Install packages
import pandas as pd
import numpy as np
import os
import matplotlib.pyplot as plt
import glob
import sys

# Make the local `code` package importable. `code/` must be a
# subfolder of the current working directory (true if you ran the clone
# cell above, or if you're running this notebook from the repo root
# locally/in Drive).
REPO_ROOT = os.getcwd()
if REPO_ROOT not in sys.path:
    sys.path.append(REPO_ROOT)


In [ ]:
# Read the coding sheet
from code.sheets_io import get_pygsheets_client, ReplaceSheet

gc = get_pygsheets_client()


In [ ]:
# ReplaceSheet() is now defined in code/sheets_io.py and was already
# imported above.


In [ ]:
# Load the reference data dictionary
!git clone https://github.com/GP2code/GP2-Data-Dictionary.git

In [ ]:
ref_dictionary_path = 'GP2-Data-Dictionary/GP2_Data_Dictionary_ver1.1-3.csv'  # kept as a variable for the run manifest below
ref = pd.read_csv(ref_dictionary_path)
print(ref.shape)

In [ ]:
# Cohort coding -> add the coding sheet's key
sheet = gc.open_by_key('') # copy Google sheet key and paste here
worksheet = sheet.worksheet_by_title('Sheet1')
d = worksheet.get_as_df(include_tailing_empty=False)
print(d.shape)
if d.shape[0]==0: # empty then copy ref
  dnew = ref[['Single Measure', 'Item', 'Description', 'ItemType', 'Required', 'Values']].copy()
  dnew[['File', 'KeyList', 'Operation', 'Action']]=np.nan
  # Update sheet 1
  worksheet = sheet.worksheet_by_title('Sheet1')
  worksheet.set_dataframe(dnew, start='A1',  fit=True, nan='')
  print('"Sheet1" CREATED from sheet_ref')
  d = dnew.copy()

In [ ]:
k1 = d[['Item', 'Description', 'ItemType', 'Required', 'Values']].astype(str).agg('-'.join, axis=1)
k2 = ref[['Item', 'Description', 'ItemType', 'Required', 'Values']].astype(str).agg('-'.join, axis=1)
not_in_d = ref.loc[~k2.isin(k1), ['Single Measure', 'Item', 'Description', 'ItemType', 'Required', 'Values']]
not_in_ref = d.loc[~k1.isin(k2), ['Item', 'Description', 'ItemType', 'Required', 'Values']]
print(not_in_d.shape)
print(not_in_ref.shape)

In [ ]:
d_conflict = pd.merge(not_in_d, not_in_ref, on='Item', suffixes=['_ref', ''], how='outer')

# Save the dictionary conflict
new_sheet = 'Dic_conflict'
ReplaceSheet(sheet, new_sheet, d_conflict)

## If the conflict exists between the most updated dic and the current dic....

Run the following and check the conflicts. The items used for some operations need extra attentions.

Also, items only appear in the most current dictionary should be evaluated if these items can be created from the dataset

Resolve conflicts and use the updated_Sheet1 as "Sheet 1"

In [ ]:
# Update the Sheet1
d2 = pd.merge(ref[['Single Measure', 'Item', 'Description', 'ItemType', 'Required', 'Values']],
              d[['Item', 'File', 'KeyList', 'Operation', 'Action']],
              on='Item', how='left')

# Save the updated dictionary
ReplaceSheet(sheet, 'updated_Sheet1', d2)

# Coding Sheet Set-up
1. Mount drive
2. Go to the cohort's processing folder
3. Copy raw data to the processing folder

In [ ]:
# Mount gdrive
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
# cd to the work folder and copy the raw data to the processin data
cohort = 'cohort'
datafolder = f'/content/drive/dir/raw_data/{cohort}'
workfolder = f'/content/drive/dir/procession/{cohort}'

os.chdir(workfolder)
!cp -r "{datafolder}"/data_combined.csv . # if possible, directly accessing the data is a cleaner use of the workfolder

In [ ]:
# Useful functions -> now defined in code/qc_utils.py
from code.qc_utils import checkDup, checkNull, TakeOneEntry


In [ ]:
!ls {workfolder}

# Pre-processing
Perform any data cleaning, transformation, derivation, etc. prior to running the processing portion of the notebook.

# Process the coding sheet


Numeric values have to have some value definitions such as (y>=0) & (y<=40)

*   Numeric items have to have conditions (e.g. (y>=0) & (y<=40)) in the "Values" column
*   String items can provide allowed values in the "Values" columns
*   The unnecessary columns should be deleted before being read by this script

## Create cleaning list


In [ ]:
cleaning_list = []

## Process the data

In [ ]:
worksheet = sheet.worksheet_by_title('Sheet1')
d = worksheet.get_as_df(empty_value=np.nan)


## Pre-flight: lint the coding sheet
Catch mistakes in the coding sheet *before* spending a run processing files, instead of discovering them one `try/except`-caught error at a time. This only checks the sheet's own syntax/consistency -> it doesn't touch the raw data files yet, so it's fast. Fix anything printed here before proceeding to the next section.

In [ ]:
from code.coding_sheet_lint import lint_coding_sheet, print_lint_results

lint_problems = lint_coding_sheet(d)
print_lint_results(lint_problems)


In [ ]:
from code.processing import process_coding_sheet

a, processing_errors, d = process_coding_sheet(d, cleaning_list=cleaning_list)


## Review processing errors
Items that failed to process (bad column name, malformed `map`/`der1` expression, unrecognized operation, etc.) are collected here instead of silently halting the run. Check this before trusting the merged output below — anything listed here did NOT make it into `a` and will be missing from the final file until the coding sheet is fixed and this notebook is re-run.

In [ ]:
processing_errors_df = pd.DataFrame(processing_errors)
print(f"{len(processing_errors_df)} item(s) failed to process")
processing_errors_df

## Merge all KeyList groups
Previously only the *first* KeyList group (`b[0]`) was kept as the final output, so any items keyed differently from the first one encountered (e.g. baseline/static items keyed by `participant_id` alone, sitting alongside longitudinal items keyed by `participant_id`+`visit_month`) were silently dropped from the file with no error. This now merges every KeyList group together instead of discarding all but the first.

In [ ]:
from code.merge_keylists import merge_keylist_groups

x, b = merge_keylist_groups(a)


In [ ]:
from datetime import datetime

today = datetime.today().strftime("%Y-%m-%d")

In [ ]:
# NOTE: this cell was missing from the original notebook -> output_path is
# referenced in the manifest step below but was never defined. Added here
# so the merged data is actually saved and the manifest step has a real path.
output_path = f'{cohort}_{today}.csv'
x.to_csv(output_path, index=False)
print(f"Saved merged output to {output_path} (shape={x.shape})")


In [ ]:
from code.manifest import build_manifest, save_manifest, log_run_to_sheet

manifest = build_manifest(
    cohort=cohort,
    sheet=sheet,
    ref_dictionary_path=ref_dictionary_path,
    ref=ref,
    d=d,
    processing_errors=processing_errors,
    output_path=output_path,
    output_shape=x.shape,
)

manifest_path = save_manifest(manifest, output_path)
log_run_to_sheet(sheet, manifest, new_sheet='run_log')
